In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
import math
import copy
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torch import tensor
from torch.nn.functional import pad

### Config

In [2]:
d_model=512
h=8
d_ff=2048
dropout=0.1
N=6

src_vocab, tgt_vocab = 11010, 19621
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Encoder Layer

### Model

In [12]:
from networks import SublayerConnection, MultiHeadedAttention, PositionwiseFeedForward, clones

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

### Inference

In [13]:
self_attn = MultiHeadedAttention(h, d_model)
feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

encoder_layer = EncoderLayer(d_model, self_attn, feed_forward, dropout)

In [14]:
x = torch.load("embed_result.pt")

In [15]:
x.shape

torch.Size([1, 50, 512])

In [17]:
result = encoder_layer(x, mask=None)

In [18]:
result.shape

torch.Size([1, 50, 512])

## Encoder

In [22]:
from networks import LayerNorm

In [19]:
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [23]:
self_attn = MultiHeadedAttention(h, d_model)
feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

encoder_layer = EncoderLayer(d_model, self_attn, feed_forward, dropout)
encoder = Encoder(encoder_layer, N)

In [26]:
x = torch.load("embed_result.pt")

In [27]:
x.shape

torch.Size([1, 50, 512])

In [24]:
result = encoder(x, mask=None)

In [29]:
result.shape

torch.Size([1, 50, 512])

In [30]:
torch.save(result, "memory.pt")